## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        #第一层，输入28*28
        self.W1 = tf.Variable(tf.random.normal([28*28, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        #第二层，前一层有100个，输出层10个
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        #x:[N, 28, 28]->[N, 784]
        x = tf.reshape(x, [-1, 28*28])
        #隐藏层是两步：矩阵乘法+ReLU激活
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)
        #输出层,返回未归一化的logits
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [11]:
train_data, test_data = mnist_dataset()
for epoch in range(200):#改成200个epoch
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.9874787 ; accuracy 0.76498336
epoch 1 : loss 0.9854838 ; accuracy 0.7654
epoch 2 : loss 0.98349935 ; accuracy 0.7658167
epoch 3 : loss 0.9815251 ; accuracy 0.7660667
epoch 4 : loss 0.9795613 ; accuracy 0.76641667
epoch 5 : loss 0.97760797 ; accuracy 0.76675
epoch 6 : loss 0.9756643 ; accuracy 0.7671667
epoch 7 : loss 0.97373086 ; accuracy 0.7675167
epoch 8 : loss 0.9718073 ; accuracy 0.76776665
epoch 9 : loss 0.96989375 ; accuracy 0.76811665
epoch 10 : loss 0.9679901 ; accuracy 0.7684
epoch 11 : loss 0.9660962 ; accuracy 0.7687
epoch 12 : loss 0.96421194 ; accuracy 0.7689833
epoch 13 : loss 0.96233726 ; accuracy 0.76956666
epoch 14 : loss 0.9604721 ; accuracy 0.7701
epoch 15 : loss 0.95861655 ; accuracy 0.7705167
epoch 16 : loss 0.9567703 ; accuracy 0.77085
epoch 17 : loss 0.9549335 ; accuracy 0.7712333
epoch 18 : loss 0.953106 ; accuracy 0.7715833
epoch 19 : loss 0.9512876 ; accuracy 0.7719167
epoch 20 : loss 0.9494785 ; accuracy 0.7722833
epoch 21 : loss 0.9476786 ; 